# Phase 1 — Data Foundation (Steps 1.1 → 1.3)

**Scope of this notebook**

| Step | Goal | Output |
| --- | --- | --- |
| **1.1** | Load all 14 tables with a CSV-safe parser | `lake.inventory()` — rows, columns, load time per table |
| **1.2** | Profile every column | `docs/data_dictionary.md` (generated) |
| **1.3** | Measure the join graph | integrity %, fan-out, orphans, key-path resolution, ER diagram |
| **+** | Integrate the campaign briefs | structured briefs + RFP requirement checklist |

**How this notebook is built to scale.** The notebook contains *no data logic*. Every
loader, profiler and join check lives in `src/agentiq/data/`, so Steps 1.4–1.9 and
Phases 3–8 import the same code instead of copying cells. Schema knowledge lives in
one declarative place — `src/agentiq/data/catalog.py` — and this notebook's job is to
**test those declarations against the data** and render the evidence.

> Declarations in the catalogue are hypotheses. Where measurement disagrees, the
> catalogue is wrong and must be corrected — never the other way round.

## 0 · Environment

In [1]:
from __future__ import annotations

import re
import sys
from pathlib import Path

import pandas as pd


def _project_root() -> Path:
    """Locate the repository root whether the kernel starts in / or in notebooks/."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src" / "agentiq").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the project tree.")


ROOT = _project_root()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

print(f"python  {sys.version.split()[0]}")
print(f"pandas  {pd.__version__}")
print(f"root    {ROOT}")

python  3.11.9
pandas  2.2.3
root    C:\Users\ValliappanM\Downloads\Hack Days 26


In [2]:
%load_ext autoreload
%autoreload 2

from agentiq.data import (
    CATALOG,
    KEY_PATHS,
    DataLake,
    ProjectPaths,
    briefs_frame,
    check_all_joins,
    columns_frame,
    coverage_frame,
    derive_fields,
    joins_frame,
    load_briefs,
    mermaid_er,
    paths_frame,
    profile_lake,
    render_data_dictionary,
    render_join_section,
    requirements_frame,
    tables_by_layer,
    tables_frame,
    trace_all_paths,
)

PATHS = ProjectPaths(ROOT).ensure_dirs()
PATHS

ProjectPaths(root=C:\Users\ValliappanM\Downloads\Hack Days 26)

## 1.1 · Inventory and load all 14 tables

Every read goes through `pandas.read_csv` with standard quoting — quoted fields
containing commas are expected in this dataset, so nothing is ever split positionally.
Declared categorical, boolean and date columns are typed at read time; the two large
tables (`bookings`, `ridership_actuals`) are mirrored to parquet on first read so every
later notebook reloads them in a fraction of the time.

**Exit criterion:** row count and column list printed for all 14 tables, zero parse errors.

In [3]:
for layer, names in tables_by_layer().items():
    print(f"{layer:>12}  {', '.join(names)}")
print(f"\n{len(CATALOG)} tables declared in the catalogue.")

   geography  cities, zone_demographics, locations
     network  route_stops, route_schedules, ridership_actuals, vehicles
   inventory  screens, dim_slot
     context  points_of_interest, events
  commercial  client_facts, bookings, lost_leads

14 tables declared in the catalogue.


In [4]:
lake = DataLake(PATHS.raw_data, cache_dir=PATHS.cache)

# First call reads every CSV; subsequent calls are memoised.
inventory = lake.inventory()
inventory.drop(columns=["column_names"])

,table,layer,file,rows,columns,memory_mb,load_seconds,source
0,cities,geography,cities.csv,3,6,0.00,0.01,csv
1,zone_demographics,geography,zone_demographics.csv,30,15,0.01,0.00,csv
2,locations,geography,locations.csv,910,6,0.30,0.00,csv
3,route_stops,network,route_stops.csv,2436,11,0.83,0.01,csv
4,route_schedules,network,route_schedules.csv,19838,7,5.32,0.02,csv
5,ridership_actuals,network,ridership_actuals.csv,2049632,7,430.54,0.41,parquet-cache
6,vehicles,network,vehicles.csv,854,5,0.17,0.00,csv
7,screens,inventory,screens.csv,11163,7,2.49,0.02,csv
8,dim_slot,inventory,dim_slot.csv,6,5,0.00,0.00,csv
9,points_of_interest,context,points_of_interest.csv,1375,13,0.50,0.01,csv


In [5]:
# The full column list per table — the literal 1.1 exit criterion.
for row in inventory.itertuples():
    print(f"\n{row.table}  ({row.rows:,} rows x {row.columns} cols)")
    print(f"  {row.column_names}")


cities  (3 rows x 6 cols)
  city_id, city_name, population, transit_density, market_tier, timezone

zone_demographics  (30 rows x 15 cols)
  zone_id, city_id, zone_name, resident_population, population_density_per_sqkm, median_age, pct_age_under_18, pct_age_18_34, pct_age_35_54, pct_age_55_plus, median_household_income, income_index, pct_bachelor_or_higher, dominant_occupation, daytime_population_multiplier

locations  (910 rows x 6 cols)
  location_id, city_id, name, city_zone, zone_id, location_type

route_stops  (2,436 rows x 11 cols)
  route_id, corridor_id, city_id, route_name, mode, direction, stop_sequence, location_id, is_first_stop, is_last_stop, num_stops

route_schedules  (19,838 rows x 7 cols)
  schedule_id, route_id, corridor_id, direction, day_type, start_time, estimated_ridership

ridership_actuals  (2,049,632 rows x 7 cols)
  schedule_id, route_id, city_id, date, day_of_week, is_holiday, actual_ridership

vehicles  (854 rows x 5 cols)
  vehicle_id, city_id, vehicle_typ

In [6]:
assert len(inventory) == 14, f"expected 14 tables, loaded {len(inventory)}"
assert (inventory["rows"] > 0).all(), "a table loaded zero rows"
assert (inventory["columns"] > 0).all(), "a table loaded zero columns"
print(
    f"STEP 1.1 PASS — {len(inventory)} tables, "
    f"{inventory['rows'].sum():,} rows, "
    f"{inventory['memory_mb'].sum():,.0f} MB resident, "
    f"{inventory['load_seconds'].sum():.1f}s to load."
)

STEP 1.1 PASS — 14 tables, 2,279,693 rows, 517 MB resident, 0.7s to load.


### Peek at every table

Two rows each — enough to sanity-check the typing without dumping 2M rows.

In [7]:
from IPython.display import display

for name in CATALOG:
    print(f"\n=== {name} — {CATALOG[name].grain}")
    display(lake[name].head(2))


=== cities — One row per city in the network.


,city_id,city_name,population,transit_density,market_tier,timezone
0,LH,Las Hackland,3200000,dense,premium,America/New_York
1,ACS,Accordionshire,850000,sprawling,value,America/Chicago



=== zone_demographics — One row per city zone.


,zone_id,city_id,zone_name,resident_population,population_density_per_sqkm,median_age,pct_age_under_18,pct_age_18_34,pct_age_35_54,pct_age_55_plus,median_household_income,income_index,pct_bachelor_or_higher,dominant_occupation,daytime_population_multiplier
0,LH-ZONE-001,LH,Downtown Core,107560,14480,32.6,6.7,39.0,38.5,15.8,134615,141.7,64.9,white_collar,3.39
1,LH-ZONE-002,LH,Harborfront,344279,7107,44.7,13.9,26.2,30.9,29.0,99465,104.7,43.8,mixed,1.52



=== locations — One row per physical location (stop / station / roadside point).


...

,client_id,company_name,industry,client_tier,home_city_id,active_cities,preferred_geographies,typical_campaign_budget,budget_variance_pct,campaign_frequency,avg_campaign_duration_days,bundle_affinity,negotiation_leverage,relationship_start_date,account_status
0,CLI-00001,Cinema Entertainment,entertainment,local_business,DAT,DAT,DAT:Central Yard,12900.0,0.28,one_off,25,single_screen,low,2021-10-08,active
1,CLI-00002,Care Clinics,healthcare,local_business,DAT,DAT,DAT:Ashford Row,6200.0,0.44,seasonal,8,single_screen,medium,2026-02-10,active



=== bookings — One row per booking line item: a screen x time block held for a date range. Occupancy needs the booking-expansion transform (Step 1.5).


,booking_id,deal_id,client_id,city_id,screen_id,ad_type,industry_vertical,campaign_objective,time_block_id,daypart,slots_booked_per_day,rotation_type,start_date,end_date,duration_days,booked_date,contracted_price_per_slot_per_day,line_item_value,deal_total_value,is_bundle,booking_status
0,DAT-BKG-0000001,DEAL-000001,CLI-00001,DAT,DAT-SCR-002597,Concert Tour Announcement (Frequency),entertainment,frequency,5,evening,1,single_rotation,2026-03-01,2026-03-27,27,2026-02-08,119.26,3220.02,3220.02,False,completed
1,DAT-BKG-0000002,DEAL-000002,CLI-00001,DAT,DAT-SCR-002432,Movie Release Teaser (Awareness),entertainment,awareness,4,afternoon,2,partial_rotation,2025-09-03,2025-10-03,31,2025-08-17,90.70,5623.40,16562.06,True,completed



=== lost_leads — One row per lost lead / failed negotiation.


,lead_id,client_id,company_name_raw,industry_vertical,city_id,requested_geography,anchor_screen_id,lead_source,lead_date,sales_stage_reached,lost_date,requested_start_date,requested_duration_days,requested_num_screens,indicated_budget,quoted_price_per_slot_per_day,client_target_price_per_slot_per_day,price_gap_pct,negotiation_rounds,competitor_mentioned,loss_reason,loss_reason_detail,campaign_objective,ad_type
0,LEAD-000001,CLI-00063,NaN,healthcare,LH,LH:East Commons,LH-SCR-002131,repeat_client_inquiry,2025-10-29,contract_sent,2025-12-23,2026-01-16,87,1,152659.64,81.19,78.10,0.0396,3,True,contract_terms_disagreement,Disagreement over cancellation/make-good terms,conversion,New Clinic Announcement (Conversion)
1,LEAD-000002,CLI-00502,NaN,hospitality,DAT,DAT:Lakeside Loop,DAT-SCR-002830,referral,2026-06-14,negotiating,2026-06-22,2026-07-05,23,9,4111.23,58.38,40.15,0.4540,3,False,price_too_high,Client benchmarked against a cheaper media mix,frequency,Loyalty Membership Drive (Frequency)


## 1.2 · Build the data dictionary

For every table: grain, row count, primary-key uniqueness, duplicate rows. For every
column: dtype, null %, cardinality, min/max (or top-10 values) and a **stated meaning**.

Meanings come from `catalog.py`; any column without one is emitted with `#TODO-semantics`,
which is the checklist that must be empty before Phase 3.

The dictionary is *generated*, so it cannot drift from the data — regenerate with
`python scripts/build_data_dictionary.py`.

In [8]:
profiles = profile_lake(lake)

table_summary = tables_frame(profiles)
table_summary

,table,layer,rows,columns,memory_mb,primary_key,pk_unique,pk_nulls,duplicate_rows,undocumented_columns,grain
0,cities,geography,3,6,0.00,city_id,True,0,0,0,One row per city in the network.
1,zone_demographics,geography,30,15,0.01,zone_id,True,0,0,0,One row per city zone.
2,locations,geography,910,6,0.30,location_id,True,0,0,0,One row per physical location (stop / station / roadside point).
3,route_stops,network,2436,11,0.83,route_id + stop_sequence,True,0,0,0,"One row per (route, stop_sequence) — a route's ordered stop list."
4,route_schedules,network,19838,7,5.32,schedule_id,True,0,0,0,One row per scheduled trip (route x day_type x departure time).
5,ridership_actuals,network,2049632,7,430.54,schedule_id + date,True,0,0,0,"One row per (scheduled trip, date) with realised ridership."
6,vehicles,network,854,5,0.17,vehicle_id,True,0,0,0,One row per vehicle carrying screens.
7,screens,inventory,11163,7,2.49,screen_id,True,0,0,0,One row per physical screen — the unit that is sold.
8,dim_slot,inventory,6,5,0.00,time_block_id,True,0,0,0,One row per sellable time block of the day.
9,points_of_interest,context,1375,13,0.50,poi_id,True,0,0,0,"One row per POI, already anchored to its nearest location."


In [9]:
# Primary keys are *tested*, not trusted.
pk_failures = table_summary[table_summary["pk_unique"] == False]  # noqa: E712
if len(pk_failures):
    display(pk_failures[["table", "primary_key", "rows", "pk_nulls", "duplicate_rows"]])
    print("^ FIX catalog.py: these declared primary keys are not unique.")
else:
    print("All 14 declared primary keys are unique with no nulls in the key.")

All 14 declared primary keys are unique with no nulls in the key.


In [10]:
columns = columns_frame(profiles)
print(f"{len(columns)} columns profiled across {columns['table'].nunique()} tables.")

# Columns most likely to need a handling rule in the Step 1.7 DQ register.
columns[columns["null_pct"] > 0].sort_values("null_pct", ascending=False)[
    ["table", "column", "dtype", "null_pct", "n_unique", "note"]
]

156 columns profiled across 14 tables.


,table,column,dtype,null_pct,n_unique,note
61,screens,vehicle_id,object,0.7657,854,"Set for the 2,615 mobile screens (23.4%), null for static ones. Exactly one of locatio..."
134,lost_leads,company_name_raw,object,0.5566,269,Free-text company name for prospects with no account yet (643 rows); null on the 807 r...
148,lost_leads,client_target_price_per_slot_per_day,float64,0.4759,731,The price the client wanted; null wherever no quote was made or no counter-offer was g...
149,lost_leads,price_gap_pct,float64,0.4759,713,(quote - client target) / target. The core price-cap calibration signal: the gap at wh...
133,lost_leads,client_id,object,0.4434,346,"Existing account, where the lead came from one. Null on 643 rows — and those are exact..."
147,lost_leads,quoted_price_per_slot_per_day,float64,0.3662,889,Our quote. Null for all 531 initial_inquiry leads and only those — the lead died befor...
60,screens,location_id,object,0.2343,910,"Set for the 8,548 static screens (76.6%), null for mobile ones."
85,events,poi_id,object,0.2343,147,"Host POI where one applies; null for 23% of events (street events, parades)."
62,screens,position,category,0.1254,6,"platform / entrance_exit / left / right / top / back. Null on all 1,400 metro_rail_coa..."


In [11]:
# Constant columns carry no signal; single-valued keys are candidate join hazards.
constant = columns[(columns["n_unique"] <= 1) & (columns["non_null"] > 0)]
display(constant[["table", "column", "n_unique", "top_values"]])

todo = columns[columns["note"] == "#TODO-semantics"]
print(f"\n{len(todo)} columns still lack a stated meaning (must be 0 before Phase 3):")
for table, group in todo.groupby("table"):
    print(f"  {table}: {', '.join(group['column'])}")

,table,column,n_unique,top_values



0 columns still lack a stated meaning (must be 0 before Phase 3):


### Categorical vocabularies

These value sets are what Phase 4 must resolve brief language *onto*, and what
`config/taxonomy.yaml` has to enumerate. Read them before inventing any label.

In [12]:
vocab = columns[(columns["kind"] == "categorical") & (columns["n_unique"] <= 30)]
for row in vocab.itertuples():
    print(f"{row.table}.{row.column}  ({row.n_unique})\n    {row.top_values}\n")

cities.city_id  (3)
    LH (1); ACS (1); DAT (1)

cities.city_name  (3)
    Las Hackland (1); Accordionshire (1); DA Town (1)

cities.transit_density  (3)
    dense (1); mixed (1); sprawling (1)

cities.market_tier  (3)
    premium (1); standard (1); value (1)

cities.timezone  (3)
    America/Chicago (1); America/Denver (1); America/New_York (1)

zone_demographics.zone_id  (30)
    LH-ZONE-001 (1); LH-ZONE-002 (1); LH-ZONE-003 (1); LH-ZONE-004 (1); LH-ZONE-005 (1); LH-ZONE-006 (1); LH-ZONE-007 (1); LH-ZONE-008 (1); LH-ZONE-009 (1); LH-ZONE-010 (1)

zone_demographics.city_id  (3)
    LH (10); ACS (10); DAT (10)

zone_demographics.zone_name  (30)
    Downtown Core (1); Harborfront (1); Old Mill District (1); Uptown Crescent (1); Financial Row (1); Cathedral Heights (1); Riverside Junction (1); North Terminus (1); Market Quarter (1); East Commons (1)

zone_demographics.dominant_occupation  (5)
    mixed (14); white_collar (7); blue_collar (3); retail_service (3); student (3)

locations.c

## 1.3 · Map and measure the join graph

Nothing here is inferred from column names. For every declared foreign key we measure:

- **Referential integrity** — share of non-null child keys present in the parent.
- **Fan-out** — max children per parent and max parents per child ⇒ 1:1 / N:1 / N:M.
  A `fanout_trap` is a silent row-multiplier in every later aggregation.
- **Orphans** — child rows joining to nothing. **These are the cold-start population.**

In [13]:
checks = check_all_joins(lake)
joins = joins_frame(checks)
joins[
    [
        "child_table",
        "child_column",
        "parent_table",
        "child_rows",
        "child_null_pct",
        "integrity_pct",
        "orphan_rows",
        "cardinality",
        "max_parents_per_child",
        "fanout_trap",
        "status",
    ]
]

,child_table,child_column,parent_table,child_rows,child_null_pct,integrity_pct,orphan_rows,cardinality,max_parents_per_child,fanout_trap,status
0,zone_demographics,city_id,cities,30,0.0000,1.0,0,N:1,1,False,OK
1,locations,city_id,cities,910,0.0000,1.0,0,N:1,1,False,OK
2,locations,zone_id,zone_demographics,910,0.0000,1.0,0,N:1,1,False,OK
3,route_stops,city_id,cities,2436,0.0000,1.0,0,N:1,1,False,OK
4,route_stops,location_id,locations,2436,0.0000,1.0,0,N:1,1,False,OK
5,route_schedules,route_id,route_stops,19838,0.0000,1.0,0,N:N,21,True,OK
6,ridership_actuals,schedule_id,route_schedules,2049632,0.0000,1.0,0,N:1,1,False,OK
7,ridership_actuals,city_id,cities,2049632,0.0000,1.0,0,N:1,1,False,OK
8,vehicles,city_id,cities,854,0.0000,1.0,0,N:1,1,False,OK
9,vehicles,corridor_id,route_stops,854,0.0000,1.0,0,N:N,42,True,OK


In [14]:
broken = joins[joins["status"] == "BROKEN"]
if len(broken):
    display(broken[["child_table", "child_column", "parent_table", "integrity_pct", "orphan_rows", "orphan_keys"]])
    for check in checks:
        if check.status == "BROKEN":
            print(f"{check.edge}: example orphan keys -> {list(check.orphan_examples)}")
else:
    print("No edge falls below 99% referential integrity.")

surprising = [c for c in checks if c.unexpected_nulls]
print(f"\nEdges with nulls we did not declare as nullable: {len(surprising)}")
for check in surprising:
    print(f"  {check.edge}: {check.child_null_pct:.1%} null — decide the handling rule.")

No edge falls below 99% referential integrity.

Edges with nulls we did not declare as nullable: 0


In [15]:
# Fan-out traps and the aggregation each consumer must apply before merging.
for check in checks:
    if check.is_fanout_trap:
        print(
            f"{check.edge}\n"
            f"    up to {check.max_parents_per_child:,} parent rows per child key\n"
            f"    -> {check.note or 'aggregate the parent to one row per key before merging'}\n"
        )

route_schedules.route_id -> route_stops.route_id
    up to 21 parent rows per child key
    -> Parent key is non-unique (route_stops is per stop); expect high fan-out.

vehicles.corridor_id -> route_stops.corridor_id
    up to 42 parent rows per child key
    -> Parent key is non-unique; a corridor spans many route_stops rows.



### Key-path resolution

Single-edge integrity is necessary but not sufficient — what matters is whether a
*screen* can be resolved end to end to demographics, to a corridor, to POIs and to
commercial history. Each path below is traced hop by hop.

In [16]:
traces = trace_all_paths(lake, KEY_PATHS)
paths_frame(traces)

,path,start_table,subset,start_rows,resolved_rows,resolution_pct,unresolved_rows,hops,why
0,static geography: screen -> location -> zone -> city,screens,static_screens,8548,8548,1.0000,0,screen -> location → location -> zone → zone -> city,D1 static exposure: resident base and daytime multiplier for a fixed screen.
1,mobile exposure: screen -> vehicle -> corridor,screens,mobile_screens,2615,2615,1.0000,0,screen -> vehicle → vehicle -> corridor stops,D1 mobile exposure: the journey a vehicle-mounted screen travels.
2,ridership: ridership_actuals -> schedule -> route,ridership_actuals,(all rows),2049632,2049632,1.0000,0,actuals -> schedule → schedule -> route stops,Daypart exposure curve per route/corridor.
3,POI context: location -> POI,points_of_interest,(all rows),1375,1375,1.0000,0,POI -> location,"D1 POI pull, distance decay and side-of-road visibility."
4,event context: event -> location/zone,events,(all rows),367,367,1.0000,0,event -> location,Phase 6 event-surge component.
5,commercial history: booking -> screen,bookings,(all rows),191109,191109,1.0000,0,booking -> screen,Pricing training data and committed occupancy per screen.
6,slot claim: booking -> dim_slot,bookings,(all rows),191109,191109,1.0000,0,booking -> time block,How a booking claims inventory in time; defines the sellable unit.
7,client: booking -> client,bookings,(all rows),191109,191109,1.0000,0,booking -> client,Client-relationship adjustment in pricing.
8,pipeline: lead -> client,lost_leads,(all rows),1450,807,0.5566,643,lead -> client,Pipeline pressure and win-probability calibration.


In [17]:
for trace in traces:
    print(f"{trace.path.name}  (start: {trace.start_rows:,} rows)")
    for label, resolved in trace.resolved_at_hop:
        share = resolved / trace.start_rows if trace.start_rows else 0
        print(f"    {label:<28} {resolved:>12,}  ({share:.1%} of start rows)")
    if trace.unresolved_rows:
        print(f"    UNRESOLVED: {trace.unresolved_rows:,} rows -> cold-start population")
    print()

static geography: screen -> location -> zone -> city  (start: 8,548 rows)
    screen -> location                  8,548  (100.0% of start rows)
    location -> zone                    8,548  (100.0% of start rows)
    zone -> city                        8,548  (100.0% of start rows)

mobile exposure: screen -> vehicle -> corridor  (start: 2,615 rows)
    screen -> vehicle                   2,615  (100.0% of start rows)
    vehicle -> corridor stops           2,615  (100.0% of start rows)

ridership: ridership_actuals -> schedule -> route  (start: 2,049,632 rows)
    actuals -> schedule             2,049,632  (100.0% of start rows)
    schedule -> route stops         2,049,632  (100.0% of start rows)

POI context: location -> POI  (start: 1,375 rows)
    POI -> location                     1,375  (100.0% of start rows)

event context: event -> location/zone  (start: 367 rows)
    event -> location                     367  (100.0% of start rows)

commercial history: booking -> screen  (s

### Static vs mobile split

`screens.location_id` XOR `screens.vehicle_id` decides which D1 exposure model applies.
This is the single most consequential fact in the schema, so it is asserted, not assumed.

In [18]:
screens = lake["screens"]
has_location = screens["location_id"].notna()
has_vehicle = screens["vehicle_id"].notna()

split = pd.DataFrame(
    {
        "screens": [
            int((has_location & ~has_vehicle).sum()),
            int((has_vehicle & ~has_location).sum()),
            int((has_location & has_vehicle).sum()),
            int((~has_location & ~has_vehicle).sum()),
        ]
    },
    index=["static (location only)", "mobile (vehicle only)", "BOTH — ambiguous", "NEITHER — orphan"],
)
split["share"] = (split["screens"] / len(screens)).map("{:.1%}".format)
display(split)

assert (has_location ^ has_vehicle).all(), (
    "location_id XOR vehicle_id does not hold — the static/mobile exposure split needs a rule."
)
print("XOR holds: every screen is unambiguously static or mobile.")

display(
    pd.crosstab(screens["screen_type"], has_vehicle.map({True: "mobile", False: "static"}))
)

,screens,share
static (location only),8548,76.6%
mobile (vehicle only),2615,23.4%
BOTH — ambiguous,0,0.0%
NEITHER — orphan,0,0.0%


XOR holds: every screen is unambiguously static or mobile.


vehicle_id,mobile,static
screen_type,,
bus,1215,0
bus_stop,0,2157
metro_rail_coach,1400,0
metro_station,0,6391


### ER diagram (measured)

Cardinality and integrity % come from the measurements above. This diagram is reused
directly in the Phase 2 C4 component view.

In [19]:
from IPython.display import Markdown

Markdown("```mermaid\n" + mermaid_er(checks) + "\n```")

```mermaid
erDiagram
    cities ||--|{ zone_demographics : "city_id 100%"
    cities ||--|{ locations : "city_id 100%"
    zone_demographics ||--|{ locations : "zone_id 100%"
    cities ||--|{ route_stops : "city_id 100%"
    locations ||--|{ route_stops : "location_id 100%"
    route_stops }o--|{ route_schedules : "route_id 100%"
    route_schedules ||--|{ ridership_actuals : "schedule_id 100%"
    cities ||--|{ ridership_actuals : "city_id 100%"
    cities ||--|{ vehicles : "city_id 100%"
    route_stops }o--|{ vehicles : "corridor_id 100%"
    cities ||--|{ screens : "city_id 100%"
    locations ||--o{ screens : "location_id 100%"
    vehicles ||--o{ screens : "vehicle_id 100%"
    cities ||--|{ points_of_interest : "city_id 100%"
    locations ||--|{ points_of_interest : "anchor_location_id 100%"
    cities ||--|{ events : "city_id 100%"
    points_of_interest ||--o{ events : "poi_id 100%"
    locations ||--o{ events : "anchor_location_id 100%"
    cities ||--|{ client_facts : "home_city_id 100%"
    client_facts ||--|{ bookings : "client_id 100%"
    cities ||--|{ bookings : "city_id 100%"
    screens ||--|{ bookings : "screen_id 100%"
    dim_slot ||--|{ bookings : "time_block_id 100%"
    client_facts ||--o{ lost_leads : "client_id 100%"
    cities ||--|{ lost_leads : "city_id 100%"
    screens ||--o{ lost_leads : "anchor_screen_id 100%"
```

## Integration check — one wide screen view

Proof that the graph actually composes: every screen, joined to its geography,
demographics, corridor, POI context and commercial history, with **fan-out traps
aggregated rather than merged**. This frame is the input contract for Phase 3.

In [20]:
locations = lake["locations"]
zones = lake["zone_demographics"]
cities = lake["cities"]
vehicles = lake["vehicles"]
pois = lake["points_of_interest"]
bookings = lake["bookings"]

# --- static geography -------------------------------------------------------
screen_view = (
    screens.merge(
        locations[["location_id", "zone_id", "city_zone", "location_type"]],
        on="location_id",
        how="left",
    )
    .merge(
        zones[["zone_id", "zone_name", "resident_population", "income_index", "daytime_population_multiplier"]],
        on="zone_id",
        how="left",
    )
    .merge(cities[["city_id", "city_name", "market_tier", "timezone"]], on="city_id", how="left")
    .merge(vehicles[["vehicle_id", "vehicle_type", "corridor_id"]], on="vehicle_id", how="left")
)

# --- POI context: aggregate first, then merge (fan-out trap) ----------------
poi_context = pois.groupby("anchor_location_id", observed=True).agg(
    poi_count=("poi_id", "count"),
    poi_footfall_total=("est_daily_footfall", "sum"),
    poi_nearest_km=("distance_to_location_km", "min"),
)
screen_view = screen_view.merge(
    poi_context, left_on="location_id", right_index=True, how="left"
)

# --- commercial history: aggregate first, then merge (fan-out trap) ---------
booking_history = bookings.groupby("screen_id", observed=True).agg(
    bookings=("booking_id", "count"),
    revenue=("line_item_value", "sum"),
    median_price_per_slot_day=("contracted_price_per_slot_per_day", "median"),
    first_booking=("start_date", "min"),
    last_booking=("end_date", "max"),
)
screen_view = screen_view.merge(
    booking_history, left_on="screen_id", right_index=True, how="left"
)
screen_view["bookings"] = screen_view["bookings"].fillna(0).astype(int)

assert len(screen_view) == len(screens), (
    f"row multiplication: {len(screens):,} screens became {len(screen_view):,} rows"
)
print(f"screen_view: {len(screen_view):,} rows x {screen_view.shape[1]} cols — no row multiplication.")
screen_view.head(3)

screen_view: 11,163 rows x 27 cols — no row multiplication.


,screen_id,city_id,screen_type,location_id,vehicle_id,position,screen_size,zone_id,city_zone,location_type,zone_name,resident_population,income_index,daytime_population_multiplier,city_name,market_tier,timezone,vehicle_type,corridor_id,poi_count,poi_footfall_total,poi_nearest_km,bookings,revenue,median_price_per_slot_day,first_booking,last_booking
0,LH-SCR-000001,LH,bus_stop,LH-LOC-0120,NaN,top,M,LH-ZONE-005,Financial Row,bus_stop,Financial Row,202967.0,159.8,3.21,Las Hackland,premium,America/New_York,NaN,NaN,1.0,29380.0,0.384,13,195060.03,87.10,2025-08-31,2027-02-01
1,LH-SCR-000002,LH,bus_stop,LH-LOC-0120,NaN,left,S,LH-ZONE-005,Financial Row,bus_stop,Financial Row,202967.0,159.8,3.21,Las Hackland,premium,America/New_York,NaN,NaN,1.0,29380.0,0.384,11,137461.92,78.94,2025-08-28,2027-01-15
2,LH-SCR-000003,LH,bus_stop,LH-LOC-0120,NaN,right,S,LH-ZONE-005,Financial Row,bus_stop,Financial Row,202967.0,159.8,3.21,Las Hackland,premium,America/New_York,NaN,NaN,1.0,29380.0,0.384,12,153650.17,77.80,2025-09-19,2027-02-19


In [21]:
# Coverage of each downstream signal — a preview of the Step 1.7 cold-start census.
coverage = pd.DataFrame(
    {
        "screens": [
            len(screen_view),
            int(screen_view["zone_id"].notna().sum()),
            int(screen_view["corridor_id"].notna().sum()),
            int(screen_view["poi_count"].notna().sum()),
            int((screen_view["bookings"] > 0).sum()),
            int((screen_view["bookings"] == 0).sum()),
        ]
    },
    index=[
        "total",
        "resolved to a zone (demographics available)",
        "resolved to a corridor (mobile exposure available)",
        "has nearby POI context",
        "has booking history (priceable from own history)",
        "ZERO bookings -> needs the fallback ladder",
    ],
)
coverage["share"] = (coverage["screens"] / len(screen_view)).map("{:.1%}".format)
coverage

,screens,share
total,11163,100.0%
resolved to a zone (demographics available),8548,76.6%
resolved to a corridor (mobile exposure available),2615,23.4%
has nearby POI context,8548,76.6%
has booking history (priceable from own history),9939,89.0%
ZERO bookings -> needs the fallback ladder,1224,11.0%


In [22]:
# Persist the integrated view so Steps 1.4-1.7 and Phase 3 do not rebuild these joins.
artifact = PATHS.artifacts / "screen_view.parquet"
try:
    screen_view.to_parquet(artifact, index=False)
    print(f"wrote {artifact.relative_to(ROOT)}  ({artifact.stat().st_size / 1024**2:.1f} MB)")
except ImportError:
    fallback = artifact.with_suffix(".csv")
    screen_view.to_csv(fallback, index=False)
    print(f"pyarrow not installed; wrote {fallback.relative_to(ROOT)} instead")

wrote data\artifacts\screen_view.parquet  (0.3 MB)


## Campaign brief integration

The other half of the input surface. Each `.docx` is parsed **deterministically** — a
`.docx` is a zip holding `word/document.xml`, so no LLM and no heavy dependency is
involved. Two layers:

1. **Structure** (`parse_brief`) — title, header label/value pairs, numbered sections,
   RFP requirement list. Lossless: every paragraph lands somewhere or is reported.
2. **Normalisation** (`derive_fields`) — budget, duration, age band, slot request.

These become the **gold parses** that the Phase 4 LLM extractor is tested against, which
is exactly why they are produced by regex rather than by a model.

In [23]:
briefs = load_briefs(PATHS.campaigns)
print(f"{len(briefs)} brief documents parsed from {PATHS.campaigns.name}/\n")

brief_table = briefs_frame(briefs)
brief_table

6 brief documents parsed from Campaigns/



,brief,source_file,campaign,company,industry_vertical,objective,target_audience,budget_amount,duration_days,age_min,age_max,slots_requested,seconds_per_minute,n_location_requirements,n_exclusions,n_rfp_requirements,n_unresolved
0,1,campaign_1.docx,ZEPHYR EV — THE FUTURE HAS NO TAILPIPE,Voltaic Motors Inc. (Brand: Zephyr EV),AUTOMOTIVE / ELECTRIC VEHICLES,Brand Awareness & Test-Drive Bookings,Urban professionals and eco-conscious upgraders (Ages 28-50),40000.0,45,28,50,1.0,15.0,2,1,3,6
1,2,campaign_2.docx,EMBER ENERGY — IGNITE EVERY HOUR,Ember Beverages LLC,FMCG / BEVERAGES (ENERGY DRINKS),Trial & Impulse Purchase,"Gen Z and young professionals, gym-goers, night-shift workers (Ages 18-30)",12000.0,21,18,30,NaN,NaN,4,0,3,2
2,3,campaign_3.docx,LOOM & THREAD — WEAR YOUR STORY,Loom & Thread Apparel Co.,RETAIL / FASHION,Seasonal Footfall & Sale Awareness,Style-conscious shoppers (Ages 20-40),22000.0,20,20,40,NaN,NaN,3,0,3,3
3,4,campaign_4.docx,"BASIL & BLOOM — FRESH, FAST, FLAVORFUL",Basil & Bloom Fast-Casual Kitchens,FOOD & BEVERAGE / QSR,Lunch-Hour Footfall & Local Recall,Office workers and students (Ages 18-35),9000.0,15,18,35,NaN,NaN,2,1,3,4
4,5,campaign_5.docx,"SKYNIMBUS AIRLINES — FLY FURTHER, FEEL CLOSER",SkyNimbus Airlines Ltd.,TRAVEL & AVIATION,New Route Awareness & Bookings,"Frequent flyers, business and leisure travelers (Ages 28-55)",35000.0,40,28,55,NaN,NaN,3,0,3,3
5,6,campaign_6.docx,LUMIÈRE COSMETICS — GLOW ON YOUR TERMS,Lumière Cosmetics Group,BEAUTY & PERSONAL CARE,New Product Launch Awareness,"Young women, beauty-conscious commuters (Ages 18-34)",20000.0,25,18,34,NaN,NaN,3,0,3,2


In [24]:
# Which header fields is every brief guaranteed to state? -> which are required in the schema.
coverage_briefs = coverage_frame(briefs)
display(coverage_briefs)

always = [c for c in coverage_briefs.columns if coverage_briefs[c].dtype == bool and coverage_briefs[c].all()]
sometimes = [
    c
    for c in coverage_briefs.columns
    if coverage_briefs[c].dtype == bool and not coverage_briefs[c].all()
]
print(f"REQUIRED in CampaignBrief (present in all briefs): {always}")
print(f"OPTIONAL (missing in at least one):                {sometimes}")
print(f"Unparsed paragraphs (should be ~0): {coverage_briefs['unparsed_paragraphs'].sum()}")

,brief,source_file,sections,unparsed_paragraphs,Company Name,Industry Vertical,Campaign Objective,Target Audience,Campaign Budget,Campaign Duration
0,1,campaign_1.docx,5,2,True,True,True,True,True,True
1,2,campaign_2.docx,5,0,True,True,True,True,True,True
2,3,campaign_3.docx,5,0,True,True,True,True,True,True
3,4,campaign_4.docx,5,0,True,True,True,True,True,True
4,5,campaign_5.docx,5,0,True,True,True,True,True,True
5,6,campaign_6.docx,5,0,True,True,True,True,True,True


REQUIRED in CampaignBrief (present in all briefs): ['Company Name', 'Industry Vertical', 'Campaign Objective', 'Target Audience', 'Campaign Budget', 'Campaign Duration']
OPTIONAL (missing in at least one):                []
Unparsed paragraphs (should be ~0): 2


In [25]:
# Hard constraints vs soft preferences, per brief.
for brief in briefs:
    fields = derive_fields(brief)
    print(f"\n{'=' * 100}\nBRIEF {fields.brief_number}: {fields.campaign_title}")
    print(f"  company     {fields.company}  |  vertical {fields.industry_vertical}")
    print(f"  objective   {fields.objective}")
    print(f"  audience    {fields.target_audience}  (ages {fields.age_min}-{fields.age_max})")
    print(f"  budget      {fields.budget_amount:,.0f}" if fields.budget_amount else "  budget      —")
    print(f"  duration    {fields.duration_days} days")
    print(f"  slots       {fields.slots_requested} slot(s), {fields.seconds_per_minute}s per minute")
    print("  HARD exclusions:")
    for exclusion in fields.exclusions or ["(none stated)"]:
        print(f"    - {exclusion}")
    print("  Location / environment requirements:")
    for requirement in fields.location_requirements or ["(none stated)"]:
        print(f"    - {requirement}")
    print("  Capabilities this brief demands:")
    for unresolved in fields.unresolved_requirements or ["(none flagged)"]:
        print(f"    ! {unresolved}")


BRIEF 1: ZEPHYR EV — THE FUTURE HAS NO TAILPIPE
  company     Voltaic Motors Inc. (Brand: Zephyr EV)  |  vertical AUTOMOTIVE / ELECTRIC VEHICLES
  objective   Brand Awareness & Test-Drive Bookings
  audience    Urban professionals and eco-conscious upgraders (Ages 28-50)  (ages 28-50)
  budget      40,000
  duration    45 days
  slots       1 slot(s), 15s per minute
  HARD exclusions:
    - Exclude bus-rear screens and value-tier inventory in high-density residential areas — the campaign is intentionally not mass-market and should avoid diluting the premium positioning.
  Location / environment requirements:
    - High-Dwell Business-District Platforms: Metro platform boards in the city's primary commercial and financial districts, where affluent professional commuters typically spend three to six minutes waiting on the platform.
    - Auto-Retail Arterial Corridors: Roadside-adjacent transit screens on major arterial routes with a dense concentration of car dealerships, positioned to

In [26]:
# The stated RFP deliverables — this table becomes the Phase 8 acceptance checklist.
requirements_frame(briefs)

,brief,source_file,campaign,requirement_no,requirement
0,1,campaign_1.docx,ZEPHYR EV — THE FUTURE HAS NO TAILPIPE,1,A curated shortlist of screens across the premium business-district platforms and auto...
1,1,campaign_1.docx,ZEPHYR EV — THE FUTURE HAS NO TAILPIPE,2,"An optimal price recommendation reflecting premium-node demand, platform dwell time, a..."
2,1,campaign_1.docx,ZEPHYR EV — THE FUTURE HAS NO TAILPIPE,3,"Projected weekly impressions and test-drive-booking potential, with logical justificat..."
3,2,campaign_2.docx,EMBER ENERGY — IGNITE EVERY HOUR,1,An inventory package combining bus-rear screens on nightlife corridors with campus-edg...
4,2,campaign_2.docx,EMBER ENERGY — IGNITE EVERY HOUR,2,Dynamic pricing reflecting late-night demand patterns and event-night surge behaviour ...
5,2,campaign_2.docx,EMBER ENERGY — IGNITE EVERY HOUR,3,"A reach plan optimised for unique late-night impressions within the stated budget, pri..."
6,3,campaign_3.docx,LOOM & THREAD — WEAR YOUR STORY,1,An inventory plan anchored on premium mall entry points and high-street retail corridors.
7,3,campaign_3.docx,LOOM & THREAD — WEAR YOUR STORY,2,Pricing reflecting weekend-weighted delivery and premium mall-entry positioning.
8,3,campaign_3.docx,LOOM & THREAD — WEAR YOUR STORY,3,"A footfall-oriented reach projection for the 20-day sale window, with the weekend-vers..."
9,4,campaign_4.docx,"BASIL & BLOOM — FRESH, FAST, FLAVORFUL",1,A tightly-scoped inventory list limited to screens within realistic walking distance o...


### Do the briefs' vocabulary and the data's vocabulary agree?

The first real integration test between the two input sources. A brief's industry
vertical must land on a value that actually exists in `bookings.industry_vertical` /
`client_facts.industry`, and its objective on a real `campaign_objective` — otherwise
Phase 4's resolution step has nothing to bind to.

In [27]:
data_verticals = sorted(
    set(bookings["industry_vertical"].astype(str).unique())
    | set(lake["client_facts"]["industry"].astype(str).unique())
)
data_objectives = sorted(bookings["campaign_objective"].astype(str).unique())

print(f"data verticals ({len(data_verticals)}): {data_verticals}")
print(f"data objectives ({len(data_objectives)}): {data_objectives}\n")


def _bind(candidates: list[str], text: str) -> list[str]:
    """Exact-token or prefix match only. Anything looser belongs in taxonomy.yaml."""
    words = set(re.findall(r"[a-z]+", text.lower()))
    return [
        value
        for value in candidates
        if value in words or any(word.startswith(value) for word in words)
    ]


rows = []
for brief in briefs:
    fields = derive_fields(brief)
    rows.append(
        {
            "brief": fields.brief_number,
            "brief_vertical": fields.industry_vertical,
            "vertical_binds_to": _bind(data_verticals, fields.industry_vertical),
            "brief_objective": fields.objective,
            "objective_binds_to": _bind(data_objectives, fields.objective),
        }
    )

mapping = pd.DataFrame(rows)
display(mapping)

unbound_v = mapping.loc[mapping["vertical_binds_to"].str.len() == 0, "brief_vertical"].tolist()
unbound_o = mapping.loc[mapping["objective_binds_to"].str.len() == 0, "brief_objective"].tolist()
print(f"Verticals with no binding ({len(unbound_v)}): {unbound_v}")
print(f"Objectives with no binding ({len(unbound_o)}): {unbound_o}")
print(
    "\nThese are the finding, not a bug: each unbound value is a synonym\n"
    "config/taxonomy.yaml must supply before Step 4.2 can resolve a brief."
)

data verticals (13): ['auto', 'cpg', 'education', 'entertainment', 'finance', 'government', 'healthcare', 'hospitality', 'nonprofit', 'real_estate', 'retail', 'technology', 'telecom']
data objectives (4): ['awareness', 'conversion', 'frequency', 'reach']



,brief,brief_vertical,vertical_binds_to,brief_objective,objective_binds_to
0,1,AUTOMOTIVE / ELECTRIC VEHICLES,[auto],Brand Awareness & Test-Drive Bookings,[awareness]
1,2,FMCG / BEVERAGES (ENERGY DRINKS),[],Trial & Impulse Purchase,[]
2,3,RETAIL / FASHION,[retail],Seasonal Footfall & Sale Awareness,[awareness]
3,4,FOOD & BEVERAGE / QSR,[],Lunch-Hour Footfall & Local Recall,[]
4,5,TRAVEL & AVIATION,[],New Route Awareness & Bookings,[awareness]
5,6,BEAUTY & PERSONAL CARE,[],New Product Launch Awareness,[awareness]


Verticals with no binding (4): ['FMCG / BEVERAGES (ENERGY DRINKS)', 'FOOD & BEVERAGE / QSR', 'TRAVEL & AVIATION', 'BEAUTY & PERSONAL CARE']
Objectives with no binding (2): ['Trial & Impulse Purchase', 'Lunch-Hour Footfall & Local Recall']

These are the finding, not a bug: each unbound value is a synonym
config/taxonomy.yaml must supply before Step 4.2 can resolve a brief.


## Generate `docs/data_dictionary.md`

Steps 1.2 and 1.3 land in one committed, regenerable document.

In [28]:
from datetime import datetime, timezone

join_section = render_join_section(checks, traces)
dictionary = render_data_dictionary(
    profiles,
    join_section=join_section,
    generated_at=datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC"),
)

target = PATHS.docs / "data_dictionary.md"
target.write_text(dictionary, encoding="utf-8")
print(f"wrote {target.relative_to(ROOT)}  ({len(dictionary):,} chars)")

wrote docs\data_dictionary.md  (49,167 chars)


## Phase 1.1–1.3 exit checklist

Run the cell below. Anything printed as `FAIL` blocks Step 1.4.

In [29]:
gates = {
    "1.1 all 14 tables load with zero parse errors": len(inventory) == 14 and (inventory["rows"] > 0).all(),
    "1.2 every declared primary key is unique": bool((table_summary["pk_unique"] != False).all()),  # noqa: E712
    "1.2 every column profiled": len(columns) == int(inventory["columns"].sum()),
    "1.2 every column has a stated meaning": len(todo) == 0,
    "1.2 data dictionary written": target.is_file(),
    "1.3 every declared edge measured": len(joins) > 0 and joins["integrity_pct"].notna().all(),
    "1.3 no edge below 99% integrity": bool((joins["integrity_pct"] >= 0.99).all()),
    "1.3 fan-out traps documented": all(
        check.note for check in checks if check.is_fanout_trap
    ),
    "1.3 screens integrate without row multiplication": len(screen_view) == len(screens),
    "+   all 6 briefs parsed with a budget": len(briefs) == 6 and bool(brief_table["budget_amount"].notna().all()),
    "+   every brief yields its RFP deliverables": bool(brief_table["n_rfp_requirements"].gt(0).all()),
    "+   every brief states all 6 header fields": bool(
        coverage_briefs[list(coverage_briefs.select_dtypes("bool").columns)].all().all()
    ),
}

width = max(len(name) for name in gates)
for name, passed in gates.items():
    print(f"{'PASS' if passed else 'FAIL'}  {name:<{width}}")

open_items = sum(1 for passed in gates.values() if not passed)
print(f"\n{len(gates) - open_items}/{len(gates)} gates passed.")
if len(todo):
    print(f"Carry-forward: {len(todo)} columns still marked #TODO-semantics (resolve before Phase 3).")

PASS  1.1 all 14 tables load with zero parse errors   
PASS  1.2 every declared primary key is unique        
PASS  1.2 every column profiled                       
PASS  1.2 every column has a stated meaning           
PASS  1.2 data dictionary written                     
PASS  1.3 every declared edge measured                
PASS  1.3 no edge below 99% integrity                 
PASS  1.3 fan-out traps documented                    
PASS  1.3 screens integrate without row multiplication
PASS  +   all 6 briefs parsed with a budget           
PASS  +   every brief yields its RFP deliverables     
PASS  +   every brief states all 6 header fields      

12/12 gates passed.


## What Step 1.4 inherits from here

| Artifact | Used by |
| --- | --- |
| `DataLake` (+ parquet cache) | every later notebook and engine — no reloading logic gets rewritten |
| `docs/data_dictionary.md` | cited by Phases 3–7 as the schema source of truth |
| `data/artifacts/screen_view.parquet` | Step 1.4 inventory shape, Phase 3 exposure models |
| measured `JoinCheck`s | the required aggregation before any merge |
| `KEY_PATHS` traces | cold-start sizing (Step 1.7), fallback-ladder design (Step 6.5) |
| gold brief parses | Phase 4 extractor test fixtures |
| RFP requirement table | Phase 8 acceptance criteria |

**Next (Step 1.4)** — profile the inventory: screens by city / type / position / size,
and the exact sellable-unit count (screens × time blocks × rotation slots × days), which
the Phase 7 solver design must cite.